### A simple test
Run the cells top to bottom; edit the `👈` values and re-run.

In [ ]:
# (hide matplotlib's harmless one-time "building the font cache" message)
import logging; logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as sps
import scipy.integrate as spi

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11, "axes.titlesize": 12,
    "axes.titleweight": "semibold", "axes.linewidth": 1.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "legend.frameon": False,
})

# Observe N samples from a Gaussian, then infer the mean.
true_mu    = 0     # 👈 true mean of the data
true_sigma = 1     # 👈 true standard deviation
N          = 50    # 👈 number of samples

data = sps.norm.rvs(true_mu, true_sigma, size=N)

# We assume we know sigma = 1, but are unsure of the mean (anywhere in -1..1).
def uniform_prior(mu):            # uninformative prior on the mean
    return sps.uniform.pdf(mu, loc=-1, scale=2)

def likelihood(data, mu):         # likelihood of the data given a mean
    return sps.norm.pdf(data, loc=mu, scale=1)

def posterior(mu, data):          # posterior is proportional to prior x likelihood
    return uniform_prior(mu) * np.prod(likelihood(data, mu))

# the data, with a few candidate means overlaid
xs = np.linspace(-4, 4, 200)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(data, bins=10, density=True, color="#4c6ef5", alpha=0.7, edgecolor="white")
for m in np.linspace(-1, 1, 5):
    ax.plot(xs, likelihood(xs, m), "k", lw=1)
ax.set(xlabel="Data value", ylabel="Density", title="Data + candidate means")
plt.show()


In [ ]:
# prior, likelihood, and posterior over 100 possible means
mus = np.linspace(-1, 1, 100)
fig, ax = plt.subplots(3, 1, figsize=(6, 7), constrained_layout=True)
ax[0].plot(mus, uniform_prior(mus), "k", lw=2); ax[0].set(ylabel="prior")
lhs = np.array([np.prod(likelihood(data, m)) for m in mus])
ax[1].plot(mus, lhs, "k", lw=2); ax[1].set(ylabel="likelihood")
post = np.array([posterior(m, data) for m in mus])
ax[2].plot(mus, post, "k", lw=2); ax[2].set(xlabel="mu", ylabel="posterior")
plt.show()


In [ ]:
# hypothesis tests from the posterior
pH0 = posterior(0, data)                             # p(mean = 0)
pH1, _ = spi.quad(posterior, 0, 1, args=(data,))     # p(mean in 0..1) by numerical integration
print(f"posterior at mean = 0     : {pH0:.4g}")
print(f"posterior for mean in 0..1: {pH1:.4g}")
print(f"Bayes factor  BF01        : {pH0 / pH1:.3g}")
